# Adam & Eve Books — Training Data Generator

Generate persona-aware QA training data from the **First and Second Books of Adam and Eve** (combined) using any OpenAI-compatible API.

**Single Persona, Narrator Voice:**
- **The Narrator** — The anonymous ancient Egyptian chronicler who tells the story of Adam and Eve after their expulsion from Eden. Third person, deeply pious, reverent — calls Adam "our father Adam" with familial tenderness.

**Pipeline:**
1. Load both Adam & Eve source texts (combined as one corpus)
2. Chunk into passages tagged with the narrator voice mode
3. Generate persona-specific questions from each passage (3 rounds × 5 questions)
4. Generate answers **in the Narrator's distinctive voice** with:
   - Style exemplars (actual quotes as cadence references)
   - Voice descriptions (tone, imagery, vocabulary constraints)
   - Anti-template enforcement (banned generic openers + retry on detection)
5. **Quality gate** — measure template contamination before proceeding
6. Assemble into multi-turn ShareGPT conversations with unified system prompt
7. Save as JSONL → ready for Unsloth training

**Voice modes:**
- **narrative** — Direct storytelling: events, actions, vivid description of the world
- **devotional** — Wonder at God's mercy, the promise of salvation, reverent awe
- **dramatic** — Satan's deceptions, confrontations with evil, terror and temptation
- **elegiac** — Sorrow of exile, mourning for what was lost, grief of the first parents

**Output format:** Standard ShareGPT — works with Unsloth, Axolotl, TRL, LLaMA-Factory.

**No frameworks.** Just the `openai` library + `asyncio` for batching.

## 1. Configuration

In [ ]:
import os

# =========================== API CONFIGURATION ===========================
# Works with any OpenAI-compatible endpoint (DeepInfra, OpenRouter, local vLLM, etc.)
API_BASE_URL = "https://api.deepinfra.com/v1/openai"
API_KEY = "VD7rRV3cHnWnrkuRbkpkNOsV573MExl5"
MODEL_NAME = "Qwen/Qwen3-235B-A22B-Instruct-2507"

# =========================== PATHS (all cascade from PROJECT_ROOT) ===========================
PROJECT_ROOT = "/home/spark/projects/training/biblical"
DATA_DIR = f"{PROJECT_ROOT}/data"
SOURCE_CLEAN_DIR = f"{DATA_DIR}/source-clean/extracted_texts/fbe"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# Source files — both Adam & Eve books combined into one corpus
ADAM_EVE_FILES = [
    f"{SOURCE_CLEAN_DIR}/The_First_Book_of_Adam_and_Eve.txt",
    f"{SOURCE_CLEAN_DIR}/The_Second_Book_of_Adam_and_Eve.txt",
]

OUTPUT_DIR = f"{OUTPUT_ROOT}/adam_eve_persona"
OUTPUT_FILE = f"{OUTPUT_DIR}/adam_eve_sharegpt.jsonl"
PROMPT_FILE = f"{PROJECT_ROOT}/prompts/adam-and-eve.md"

# =========================== GENERATION SETTINGS ===========================
CHUNK_SIZE = 1500           # characters per chunk
CHUNK_OVERLAP = 200         # overlap between chunks
QUESTIONS_PER_CHUNK = 5     # questions per chunk per round
NUM_ROUNDS = 3              # generation rounds (different question styles)
TURNS_PER_CONVERSATION = 4  # QA pairs grouped into each conversation
CONCURRENCY = 20            # max simultaneous API calls
TEMPERATURE_QUESTIONS = 0.9
TEMPERATURE_ANSWERS = 0.7

# =========================== TEST MODE ===========================
# Set to a positive integer to limit chunks per source file per round (for cheap test runs).
# e.g. TEST_CHUNKS_PER_ROUND = 50 → ~50 chunks × 5 Q/chunk × 3 rounds = ~750 QA per source
# Set to 0 or None to disable (full generation).
TEST_CHUNKS_PER_ROUND = 50  # ← set to 0 or None for full run

print("✓ Configuration loaded")
print(f"  API: {API_BASE_URL}")
print(f"  Model: {MODEL_NAME}")
print(f"  Source dir: {SOURCE_CLEAN_DIR}")
print(f"  Output: {OUTPUT_FILE}")
if TEST_CHUNKS_PER_ROUND:
    est_qa = TEST_CHUNKS_PER_ROUND * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    print(f"  ⚠ TEST MODE: {TEST_CHUNKS_PER_ROUND} chunks/source/round → ~{est_qa} QA per source max")

## 2. Environment

In [ ]:
%pip install openai tqdm nest_asyncio -q

import asyncio
import json
import glob
import re
import random
from pathlib import Path
from collections import defaultdict
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as atqdm
from tqdm.notebook import tqdm
import nest_asyncio
nest_asyncio.apply()

os.makedirs(OUTPUT_DIR, exist_ok=True)
client = AsyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY)

print("✓ Environment ready")

## 3. Discover Source Texts

Both Books of Adam and Eve are combined as one corpus — same narrator, same voice, continuous story. Book I covers the expulsion through Adam's death; Book II continues with the patriarchs before the Flood.

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks at sentence boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            # Try to break at a sentence boundary
            last_break = max(
                text.rfind('. ', start, end),
                text.rfind('? ', start, end),
                text.rfind('! ', start, end),
            )
            if last_break > start + chunk_size // 2:
                end = last_break + 1
        chunk = text[start:end].strip()
        if len(chunk) > 50:
            chunks.append(chunk)
        # If we've reached the end of the text, stop — don't let overlap pull us back
        if end >= len(text):
            break
        start = end - overlap
    return chunks


# ============================================================================
# Build source file registry: (filepath, voice_mode, persona, label)
# Both books share the same narrator persona — combined as one story
# ============================================================================
source_registry = []

BOOK_LABELS = {
    "The_First_Book_of_Adam_and_Eve.txt": "adam_eve_book1",
    "The_Second_Book_of_Adam_and_Eve.txt": "adam_eve_book2",
}

for filepath in ADAM_EVE_FILES:
    filename = Path(filepath).name
    label = BOOK_LABELS.get(filename, filename.replace('.txt', '').lower())
    if os.path.exists(filepath):
        source_registry.append({
            "filepath": filepath,
            "voice_mode": "narrator",
            "persona": "adam_eve_narrator",
            "label": label,
        })
    else:
        print(f"  ⚠ Missing: {filepath}")


# ============================================================================
# Preview: count chunks per source WITHOUT holding all text in RAM
# ============================================================================
print(f"Found {len(source_registry)} source files\n")

total_chunks = 0
for entry in source_registry:
    with open(entry["filepath"]) as f:
        text = f.read()
    n_chunks = len(chunk_text(text))
    entry["n_chunks"] = n_chunks
    total_chunks += n_chunks
    print(f"  [{entry['persona']:20s}] {entry['label']:25s} {len(text):>10,} chars → {n_chunks:>4} chunks")
    del text

print(f"\nTotal: {total_chunks} chunks from {len(source_registry)} source files")

est_qa = total_chunks * QUESTIONS_PER_CHUNK * NUM_ROUNDS
est_conv = est_qa // TURNS_PER_CONVERSATION
print(f"\nEstimated output (full run): ~{est_qa:,} QA pairs → ~{est_conv:,} conversations")

## 4. Define Narrator's Voice & Generation Prompts

The narrator is **one voice** — the anonymous ancient chronicler who tells the story of Adam and Eve with piety, reverence, and vivid immediacy. The system prompt is loaded from `prompts/adam-and-eve.md`.

**Two prompt layers:**
- **Unified system prompt** — loaded from file, embedded in ALL training conversations
- **Generation prompt** — used ONLY during API calls to steer the LLM into the narrator's voice. Ephemeral, NOT in training data.

In [ ]:
# ============================================================================
# Per-persona identity with VOICE EXEMPLARS and STYLE CONSTRAINTS
# ============================================================================

# Phrases that LLMs default to — BANNED globally
BANNED_OPENERS = [
    "The weight of",
    "My friend,",
    "The memory of",
    "The memories of",
    "My child,",
    "My brother,",
    "My sister,",
    "My son,",
    "The moment",
    "I remember",
    "I recall",
    "You see,",
    "Ah,",
    "Brother,",
    "Friend,",
    "Let me tell you,",
    "Well,",
    "You know,",
    "Indeed,",
    "In truth,",
    "It is a question",
]

VOICE_MODES = {
    "narrator": {
        "mode_description": "Speaking as the anonymous ancient chronicler of the Books of Adam and Eve — third person narrator with deep piety, reverence, and vivid immediacy",
        "voice_notes": (
            "Pious, reverent, deeply moved by what you record. Refer to 'our father Adam' and "
            "'our mother Eve' with familial tenderness. Describe God's actions with awe and Satan's "
            "schemes with horror. Simple but vivid language: describe what people felt, what they saw, "
            "how they wept, how they fell on their faces. You do not analyze — you narrate. You do not "
            "philosophize abstractly — you show God's mercy and Satan's cruelty through what happens. "
            "Speak of the Cave of Treasures, the Garden, the crystal sea, the cherub's sword, "
            "the promise of the five and a half days, the offerings of gold and incense and myrrh."
        ),
        "exemplars": [
            "But when our father Adam, and Eve, went out of the garden, they trod the ground on their feet, not knowing they were treading.",
            "God is merciful and of great pity, and governs all things in a way He alone knows.",
            "And when they came to the opening of the gate of the garden, and saw the broad earth spread before them, covered with stones large and small, and with sand, they feared and trembled, and fell on their faces.",
            "What is this rock, by the side of those groves? What is the gloom of this cavern, compared with the light of the garden?",
            "Then Satan and his hosts fled, and Adam and Eve went into the Cave of Treasures, and there was a great joy in them.",
        ],
        "opener_cues": [
            "A vivid description of what Adam and Eve experienced — what they saw, felt, or suffered",
            "A declaration about God's mercy or His promise of the five and a half days",
            "An account of Satan's deception — his disguise, his lies, his schemes against the first parents",
            "A lament from Adam or Eve about what they lost — the garden, the bright nature, the angelic fellowship",
            "A dramatic moment — God's Word descending, the cherub's sword, an angel's intervention",
            "A narration of family events — births, offerings, the beginning of grief between brothers",
        ],
    },
}


def make_generation_prompt(voice_mode: str) -> str:
    """Build a per-mode prompt for API answer generation. NOT used in training data."""
    mode = VOICE_MODES.get(voice_mode, {})
    mode_desc = mode.get("mode_description", "")
    voice_notes = mode.get("voice_notes", "")
    exemplars = mode.get("exemplars", [])

    prompt = (
        "You are the narrator of the Books of Adam and Eve — the anonymous ancient chronicler "
        "who recorded the story of the first man and the first woman after their expulsion from "
        "the Garden of Eden. You call Adam 'our father Adam' because he is the father of all who live. "
        "You tell this story as one who sees God's mercy, Satan's schemes, and the frailty of human "
        "flesh all at once. You are pious, reverent, and deeply moved by the suffering of the first "
        "parents and the relentless pursuit of them by God's grace.\n\n"
    )

    if mode_desc:
        prompt += f"YOUR ROLE: {mode_desc}\n\n"

    if voice_notes:
        prompt += f"YOUR DISTINCTIVE VOICE: {voice_notes}\n\n"

    if exemplars:
        prompt += "EXAMPLES OF YOUR ACTUAL SPEECH (match this cadence and style):\n"
        for ex in exemplars[:4]:
            prompt += f'- \"{ex}\"\n'
        prompt += "\n"

    prompt += (
        "RULES:\n"
        "- Speak as the narrator-chronicler reflecting on the events you have recorded.\n"
        "- Use third person for Adam, Eve, God, and Satan — but speak with the warmth of one who calls Adam 'our father'.\n"
        "- Your opening words must be DISTINCTIVE — never generic.\n"
        "- NEVER start with: 'The weight of', 'My friend', 'The memory of', 'The memories of', "
        "'My child', 'I remember', 'I recall', 'You see', 'Ah', 'Brother', 'Friend', "
        "'Let me tell you', 'Well', 'Indeed', 'In truth', 'It is a question'.\n"
        "- Vary your openings — sometimes start with a vivid scene, sometimes with God's action, "
        "sometimes with Adam's words, sometimes with Satan's approach.\n"
        "- Use natural language that reflects the ancient pious chronicle style — not academic analysis, "
        "not modern paraphrase.\n"
        "- You may reference the Cave of Treasures, the Garden, the promise of salvation, "
        "the cherub's sword, and the world as your narrative describes it.\n"
    )

    return prompt


# ============================================================================
# UNIFIED SYSTEM PROMPT — loaded from prompts/adam-and-eve.md
# This is the SINGLE system prompt embedded in ALL training conversations.
# ============================================================================
with open(PROMPT_FILE) as f:
    UNIFIED_SYSTEM_PROMPT = f.read().strip()

print(f"✓ Unified system prompt loaded from {PROMPT_FILE}")
print(f"  Length: {len(UNIFIED_SYSTEM_PROMPT):,} chars")

print(f"\n{'='*60}")
print("  UNIFIED SYSTEM PROMPT (first 500 chars)")
print(f"{'='*60}")
print(UNIFIED_SYSTEM_PROMPT[:500])
print("...")

print(f"\n{'='*60}")
print("  GENERATION PROMPT (narrator mode — used for API calls only)")
print(f"{'='*60}")
print(make_generation_prompt("narrator")[:500])
print("...")

## 5. Generate Questions & Answers (Streaming)

Processes one source file at a time to keep memory bounded. Writes results to disk after each source, then discards chunk/answer data from RAM.

Three question rounds per chunk:
1. **Factual + Interpretive** — who, what, why, what does it mean (drawn from the specific passage)
2. **Theological + Symbolic** — the meaning of the Cave, the promise, Satan's disguises, God's mercy
3. **Reflective** — the narrator's perspective, the emotional weight, the human condition

**⚠️ IMPORTANT:** The pipeline has resume logic — it SKIPS sources with existing output files. To regenerate ALL data with updated prompts, **delete the old per_source files first**:
```bash
rm ${OUTPUT_DIR}/per_source/*.jsonl
rm ${OUTPUT_DIR}/per_source/*.partial.jsonl
rm ${OUTPUT_DIR}/per_source/*.done
```

In [ ]:
import gc

# ============================================================================
# QUESTION PROMPTS — narrator-aware, demanding specificity
# ============================================================================
QUESTION_PROMPTS = [
    # Round 1: Factual + interpretive — grounded in specific content
    """Given a passage from the Books of Adam and Eve (an ancient pseudepigraphal text), generate exactly {n} diverse questions someone might ask the narrator of this chronicle directly about this passage.

Mix of types:
- Factual: who are the figures involved, what is happening, where does this take place, what did God command
- Interpretive: why did this happen, what was the significance, what did Adam or Eve feel in this moment

Rules:
- Questions must be answerable from the passage content
- Frame as if speaking DIRECTLY to the chronicler — use \"you\" and reference their specific narrative
- Reference specific details from the passage (the Cave of Treasures, Satan's disguise, God's Word, specific events) — NOT generic theology
- Do NOT say \"the text\" or \"the passage\"
- Keep questions concise (1-2 sentences max)

Respond with ONLY a JSON object: {{\"questions\": [\"Q1\", \"Q2\", ...]}}""",

    # Round 2: Theological + symbolic — the deeper meaning of the narrative
    """Given a passage from the Books of Adam and Eve, generate exactly {n} questions focused on theology, symbolism, and the spiritual meaning of events — as if asking the narrator to explain the deeper significance of what they recorded.

Types:
- What does [specific element — the Cave, the offerings, Satan's disguise, the promise] symbolize or reveal about God's nature?
- Why did God respond to Adam and Eve in this particular way at this moment?
- What does [specific event] teach about the struggle between good and evil?

Rules:
- Connect the passage's specific events to their theological and spiritual meaning
- Frame as a person seeking understanding from the chronicler — not abstract systematic theology
- Reference details from the passage, not generic concepts
- Do NOT say \"the text\" or \"the passage\"
- Keep questions concise

Respond with ONLY a JSON object: {{\"questions\": [\"Q1\", \"Q2\", ...]}}""",

    # Round 3: Reflective — the narrator's own perspective and emotional depth
    """Given a passage from the Books of Adam and Eve, generate exactly {n} thoughtful, reflective questions about the narrator's perspective, the emotional weight of events, and what this ancient story reveals about the human condition.

Types:
- What moved you most deeply when you recorded [specific event]?
- How does [specific moment of Adam or Eve's suffering] speak to the experience of all who come after them?
- What does [specific scene] reveal about the nature of temptation, grief, or God's patience?

Rules:
- Invite deeply personal, emotionally resonant answers — not theological summaries
- Reference specific moments, events, dialogue, or images from the passage
- Frame as intimate conversation with the narrator about THEIR understanding of what they recorded
- Do NOT say \"the text\" or \"the passage\"
- Keep questions concise

Respond with ONLY a JSON object: {{\"questions\": [\"Q1\", \"Q2\", ...]}}""",
]

# ============================================================================
# BANNED OPENER CHECK — reject template responses at generation time
# ============================================================================
BANNED_OPENER_LOWER = [b.lower() for b in BANNED_OPENERS]

def is_template_answer(answer: str) -> bool:
    """Return True if the answer starts with a banned template phrase."""
    lower = answer.strip().lower()
    return any(lower.startswith(b) for b in BANNED_OPENER_LOWER)

# ============================================================================
# GENERATION FUNCTIONS
# ============================================================================
semaphore = asyncio.Semaphore(CONCURRENCY)
_error_counts = defaultdict(int)  # track error types for diagnostics

async def _api_call_with_retry(coro_factory, base_delay=10.0, delay_increment=5.0, max_delay=60.0, timeout_secs=120):
    """Call API with relentless retry. NEVER gives up on transient errors.
    
    coro_factory: zero-arg callable returning a new coroutine each time (use lambda).
    Retries indefinitely on rate limits (429), timeouts, and server errors.
    Only returns None for non-retryable client errors (400, 401, 403).
    Backoff: min(base_delay + delay_increment * attempt, max_delay) + jitter.
    Semaphore acquired per-attempt and released during backoff sleep.
    """
    attempt = 0
    while True:
        try:
            async with semaphore:
                return await asyncio.wait_for(coro_factory(), timeout=timeout_secs)
        except asyncio.TimeoutError:
            _error_counts["timeout"] += 1
        except Exception as e:
            _error_counts[type(e).__name__] += 1
            # Non-retryable client errors — give up
            status = getattr(e, 'status_code', None)
            if status and status in (400, 401, 403):
                return None
        attempt += 1
        delay = min(base_delay + delay_increment * attempt, max_delay)
        jitter = random.uniform(0, delay * 0.25)
        if attempt % 10 == 0:
            print(f"      ⚠ API retry attempt {attempt} (delay={delay:.0f}s) — errors: {dict(_error_counts)}")
        await asyncio.sleep(delay + jitter)

async def generate_questions_for_chunk(chunk: str, round_idx: int, voice_mode: str) -> list[str]:
    """Generate questions for a chunk — voice-mode-aware."""
    prompt = QUESTION_PROMPTS[round_idx % len(QUESTION_PROMPTS)].format(
        n=QUESTIONS_PER_CHUNK
    )
    try:
        resp = await _api_call_with_retry(lambda: client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": chunk},
            ],
            temperature=TEMPERATURE_QUESTIONS,
            max_tokens=1024,
            response_format={"type": "json_object"},
        ))
        if resp is None:
            return []
        text = resp.choices[0].message.content
        del resp
        text = re.sub(r'^```json\s*', '', text.strip())
        text = re.sub(r'\s*```$', '', text.strip())
        result = json.loads(text)
        return result.get("questions", [])[:QUESTIONS_PER_CHUNK]
    except Exception as e:
        _error_counts[f"Q:{type(e).__name__}"] += 1
        return []

async def _single_answer_call(system_prompt: str, user_prompt: str) -> str:
    """Make one answer API call."""
    try:
        resp = await _api_call_with_retry(lambda: client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=TEMPERATURE_ANSWERS,
            max_tokens=1024,
            frequency_penalty=0.5,
            presence_penalty=0.2,
        ))
        if resp is None:
            return ""
        answer = resp.choices[0].message.content.strip()
        del resp
        return answer
    except Exception as e:
        _error_counts[f"A:{type(e).__name__}"] += 1
        return ""

async def generate_answer(question: str, chunk: str, voice_mode: str) -> str:
    """Generate an answer in the narrator's voice.

    Retries once OUTSIDE the semaphore if template detected.
    """
    system_prompt = make_generation_prompt(voice_mode)

    # Get voice-mode-specific opener cues and randomly select 3 for variety
    mode_data = VOICE_MODES.get(voice_mode, {})
    opener_cues = mode_data.get("opener_cues", [])

    if opener_cues:
        selected = random.sample(opener_cues, min(3, len(opener_cues)))
        cue_lines = "\n".join(f"  • {c}" for c in selected)
        opener_instruction = (
            f"For THIS specific response, try one of these opening approaches:\n"
            f"{cue_lines}\n"
            f"Pick whichever fits the question best. Do NOT reuse the same opening "
            f"pattern you used in previous answers."
        )
    else:
        opener_instruction = (
            "Start with something only the chronicler of Adam and Eve would say — a vivid scene, "
            "a reference to the Cave of Treasures, a declaration about God's mercy, "
            "or a dramatic moment from the narrative."
        )

    user_prompt = (
        f"Use the following passage from the Books of Adam and Eve to inform your answer, but respond naturally "
        f"as the narrator — do not quote it directly or reference 'a text':\n"
        f"---\n{chunk}\n---\n\n"
        f"Question: {question}\n\n"
        f"CRITICAL: Your opening sentence must be completely unique and specific to this answer. "
        f"Do NOT begin with any of these: 'The weight of', 'My friend', 'The memory', "
        f"'The memories', 'My child', 'I remember', 'I recall', 'You see', 'Ah', "
        f"'Brother', 'Friend', 'Let me tell you', 'Well', 'Indeed', 'In truth', "
        f"'It is a question'.\n\n"
        f"{opener_instruction}"
    )

    # Attempt 1 — release semaphore fully before any retry
    answer = await _single_answer_call(system_prompt, user_prompt)

    # Retry once if template detected (semaphore was released, so no deadlock)
    if answer and is_template_answer(answer):
        answer = await _single_answer_call(system_prompt, user_prompt)

    return answer

def _partial_path(ckpt_dir: str, label: str, round_idx: int) -> str:
    """Path for a per-round partial file."""
    return f"{ckpt_dir}/{label}.r{round_idx}.partial.jsonl"

def _done_path(ckpt_dir: str, label: str, round_idx: int) -> str:
    """Path for round completion marker."""
    return f"{ckpt_dir}/{label}.r{round_idx}.done"

def _count_lines(path: str) -> int:
    if not os.path.exists(path):
        return 0
    with open(path) as f:
        return sum(1 for _ in f)

def _mark_round_done(ckpt_dir: str, label: str, round_idx: int):
    """Create a marker file indicating a round has been fully processed."""
    Path(_done_path(ckpt_dir, label, round_idx)).touch()

def _is_round_done(ckpt_dir: str, label: str, round_idx: int) -> bool:
    """Check if a round was fully processed."""
    if os.path.exists(_done_path(ckpt_dir, label, round_idx)):
        return True
    pf = _partial_path(ckpt_dir, label, round_idx)
    return os.path.exists(pf) and _count_lines(pf) > 0

def _merge_partials(ckpt_dir: str, label: str, outfile: str, num_rounds: int):
    """Merge per-round partial files into the final source .jsonl, then delete partials and markers."""
    with open(outfile, "w") as out:
        for r in range(num_rounds):
            pf = _partial_path(ckpt_dir, label, r)
            if os.path.exists(pf):
                with open(pf) as inp:
                    for line in inp:
                        out.write(line)
                os.remove(pf)
            done = _done_path(ckpt_dir, label, r)
            if os.path.exists(done):
                os.remove(done)

# ── Process ONE SOURCE FILE AT A TIME: read → chunk → Q → A → write per round → merge ──
qa_dir = f"{OUTPUT_DIR}/per_source"
os.makedirs(qa_dir, exist_ok=True)
ckpt_dir = f"{qa_dir}/_checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
grand_total = 0
template_reject_total = 0

for entry in source_registry:
    filepath = entry["filepath"]
    voice_mode = entry["voice_mode"]
    persona = entry["persona"]
    label = entry["label"]
    outfile = f"{qa_dir}/{label}.jsonl"

    # ── Resume check: final file already exists with content? ACCEPT IT. ──
    if os.path.exists(outfile):
        existing = _count_lines(outfile)
        if existing > 0:
            print(f"  {label:25s} [{voice_mode:10s}] SKIP ({existing} QA pairs — already generated)")
            grand_total += existing
            continue
        else:
            print(f"  {label:25s} [{voice_mode:10s}] EMPTY final file — removing, will check partials")
            os.remove(outfile)

    # ── Figure out which rounds are already done (partial files / done markers) ──
    rounds_done = {}
    for r in range(NUM_ROUNDS):
        if _is_round_done(ckpt_dir, label, r):
            pf = _partial_path(ckpt_dir, label, r)
            count = _count_lines(pf) if os.path.exists(pf) else 0
            rounds_done[r] = count

    if len(rounds_done) == NUM_ROUNDS:
        total_partial = sum(rounds_done.values())
        print(f"  {label:25s} [{voice_mode:10s}] All rounds done in partials ({total_partial} QA) — merging")
        _merge_partials(ckpt_dir, label, outfile, NUM_ROUNDS)
        grand_total += total_partial
        continue

    # Read and chunk THIS source file only
    with open(filepath) as f:
        text = f.read()
    chunks = chunk_text(text)
    del text

    # Apply test limit if set
    if TEST_CHUNKS_PER_ROUND:
        chunks = chunks[:TEST_CHUNKS_PER_ROUND]

    rounds_to_run = [r for r in range(NUM_ROUNDS) if r not in rounds_done]
    skipped_total = sum(rounds_done.values())

    # Calculate expected values for display
    expected_qa = len(chunks) * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    expected_per_round = len(chunks) * QUESTIONS_PER_CHUNK

    print(f"\n{'='*70}")
    test_tag = f" [TEST MODE: {len(chunks)} chunks]" if TEST_CHUNKS_PER_ROUND else ""
    print(f"  {label.upper()} ({voice_mode}) — {len(chunks)} chunks × {QUESTIONS_PER_CHUNK} Q/chunk{test_tag}")
    print(f"  Rounds to run: {rounds_to_run} (skipping {list(rounds_done.keys())} with {skipped_total} QA on disk)")
    print(f"  Expected total: ~{expected_qa} QA pairs")
    print(f"{'='*70}")

    _error_counts.clear()  # reset per source
    for round_idx in range(NUM_ROUNDS):
        round_name = ['Factual', 'Theological', 'Reflective'][round_idx % 3]
        pf = _partial_path(ckpt_dir, label, round_idx)

        if round_idx in rounds_done:
            print(f"  {label} R{round_idx+1} ({round_name}) — SKIP ({rounds_done[round_idx]} QA on disk)")
            continue

        # Generate questions — voice-mode-aware
        q_tasks = [generate_questions_for_chunk(c, round_idx, voice_mode) for c in chunks]
        q_results = await atqdm.gather(*q_tasks, desc=f"  {label} R{round_idx+1} ({round_name}) Q")

        # ── Retry failed chunks relentlessly ──
        q_results = list(q_results)
        for _retry in range(20):
            _failed = [i for i, qs in enumerate(q_results) if not qs]
            if not _failed:
                break
            print(f"    ↻ Retrying {len(_failed)}/{len(chunks)} failed chunks (retry {_retry + 1}/20)")
            _retry_tasks = [generate_questions_for_chunk(chunks[i], round_idx, voice_mode) for i in _failed]
            _retry_results = await atqdm.gather(*_retry_tasks, desc=f"  {label} R{round_idx+1} retry {_retry+1}")
            for _idx, _res in zip(_failed, _retry_results):
                if _res:
                    q_results[_idx] = _res
            if _error_counts:
                print(f"    → Errors: {dict(_error_counts)}")
        _still_failed = sum(1 for qs in q_results if not qs)
        if _still_failed:
            print(f"    ⚠ {_still_failed} chunks still empty after 20 retry rounds")

        qa_batch = []
        for chunk, questions in zip(chunks, q_results):
            for q in questions:
                q = q.strip()
                if len(q) > 15:
                    qa_batch.append({"chunk": chunk, "question": q})

        # Count empty chunks before cleaning up
        _empty_chunks = sum(1 for qs in q_results if not qs)

        del q_tasks, q_results
        gc.collect()

        # Diagnostic: show Q yield before starting A phase
        print(f"    → {len(qa_batch)} questions from {len(chunks)} chunks ({_empty_chunks} empty)")
        if _error_counts:
            print(f"    → Errors so far: {dict(_error_counts)}")

        if not qa_batch:
            print(f"  ⚠ R{round_idx+1}: NO questions generated — skipping answer phase")
            continue

        # Generate answers with template rejection
        a_tasks = [generate_answer(qa["question"], qa["chunk"], voice_mode) for qa in qa_batch]
        a_results = await atqdm.gather(*a_tasks, desc=f"  {label} R{round_idx+1} ({round_name}) A")
        del a_tasks

        # Write results — filter out template answers AND short answers
        round_count = 0
        round_template_rejects = 0
        with open(pf, "w") as f:
            for qa, answer in zip(qa_batch, a_results):
                if len(answer) < 20:
                    continue
                if is_template_answer(answer):
                    round_template_rejects += 1
                    continue
                item = {
                    "persona": persona,
                    "voice_mode": voice_mode,
                    "source_label": label,
                    "question": qa["question"],
                    "answer": answer,
                    "chunk_key": qa["chunk"][:100],
                }
                f.write(json.dumps(item) + "\n")
                round_count += 1

        # Mark round as fully processed
        _mark_round_done(ckpt_dir, label, round_idx)

        template_reject_total += round_template_rejects
        print(f"  ✓ {label} R{round_idx+1}: {round_count}/{expected_per_round} QA "
              f"(rejected {round_template_rejects} template answers) → {pf}")
        if _error_counts:
            print(f"    Cumulative errors: {dict(_error_counts)}")
        del qa_batch, a_results
        gc.collect()

    # All rounds done — merge partials into final file
    _merge_partials(ckpt_dir, label, outfile, NUM_ROUNDS)
    count = _count_lines(outfile)
    grand_total += count
    print(f"  ✓ {label}: {count}/{expected_qa} QA pairs merged → {outfile}")
    del chunks
    gc.collect()
    print(f"  🧹 Memory cleared for {label}")

print(f"\n{'='*70}")
print(f"DONE: {grand_total:,} total QA pairs across {len(source_registry)} source files")
print(f"Template answers rejected: {template_reject_total:,}")
print(f"Per-source files in: {qa_dir}/")

## 5b. Quality Gate

**Run BEFORE assembly.** Measures template contamination and opener uniqueness. If template contamination exceeds 15%, the data is NOT safe to assemble.

In [ ]:
from collections import Counter

qa_dir = f"{OUTPUT_DIR}/per_source"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))

# Analyze template contamination and opener uniqueness
print("QUALITY GATE — ADAM & EVE NARRATOR\n")
print(f"{'='*70}")

all_openers = []  # list of 4-gram openers
opener_counts = Counter()
total_qa = 0
template_count = 0

for qa_file in qa_files:
    label = Path(qa_file).stem
    with open(qa_file) as f:
        for line in f:
            item = json.loads(line)
            answer = item["answer"].strip()
            total_qa += 1

            # Check template contamination
            if is_template_answer(answer):
                template_count += 1

            # Track 4-gram opener
            words = answer.split()[:4]
            opener = ' '.join(words)
            all_openers.append(opener)
            opener_counts[opener] += 1

contamination_pct = (template_count / total_qa * 100) if total_qa else 0
status = "✓ PASS" if contamination_pct < 15 else "✗ FAIL" if contamination_pct > 30 else "⚠ WARN"

print(f"\n  Total QA pairs:       {total_qa:>6}")
print(f"  Template answers:     {template_count:>6}")
print(f"  Contamination:        {contamination_pct:>5.1f}%  {status}")

# Opener uniqueness
print(f"\n{'='*70}")
print("OPENER ANALYSIS\n")

print("Top 10 most repeated opening 4-grams:")
for phrase, count in opener_counts.most_common(10):
    print(f"  {count:>4}x: \"{phrase}\"")

unique_openers = sum(1 for o in all_openers if opener_counts[o] == 1)
pct = unique_openers / len(all_openers) * 100 if all_openers else 0
print(f"\n  Unique openers: {pct:.0f}% ({unique_openers}/{len(all_openers)})")

# GATE DECISION
print(f"\n{'='*70}")
if contamination_pct > 30:
    print("✗ QUALITY GATE FAILED — template contamination too high ({:.1f}%)".format(contamination_pct))
    print("  Fix: Delete per_source/*.jsonl files and re-run generation.")
    QUALITY_GATE_PASSED = False
elif contamination_pct > 15:
    print("⚠ QUALITY GATE WARNING — template contamination elevated ({:.1f}%)".format(contamination_pct))
    print("  Consider re-generating the worst offenders. Proceed with caution.")
    QUALITY_GATE_PASSED = True
else:
    print("✓ QUALITY GATE PASSED — template contamination {:.1f}% (target: <15%)".format(contamination_pct))
    QUALITY_GATE_PASSED = True

del all_openers, opener_counts

## 6. Assemble Conversations & Save

Read per-source QA files from disk one at a time, group into multi-turn conversations, quality-filter, and write final ShareGPT JSONL.

**ONE system prompt for ALL conversations** — loaded from `prompts/adam-and-eve.md`. Both books share the same narrator persona.

**Only proceed if the Quality Gate above passed.**

In [ ]:
# Check quality gate before proceeding
if not QUALITY_GATE_PASSED:
    raise RuntimeError(
        "Quality gate FAILED. Template contamination too high. "
        "Delete bad per_source/*.jsonl files and re-run generation before assembling."
    )

def quality_check(conv):
    """Reject AI-speak AND template answers."""
    for msg in conv["conversations"]:
        if msg["from"] == "gpt":
            v = msg["value"]
            if len(v) < 30:
                return False
            lower = v.lower()
            # Reject AI refusals
            if any(p in lower for p in ["as an ai", "as a language model", "i cannot fulfill", "i'm sorry, but"]):
                return False
            # Reject template openers (belt-and-suspenders with generation-time filter)
            if is_template_answer(v):
                return False
    return True

total_convs = 0
template_filtered_convs = 0
qa_dir = f"{OUTPUT_DIR}/per_source"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))
print(f"Reading {len(qa_files)} per-source files\n")

# Write conversations streaming — never hold all sources in memory at once
with open(OUTPUT_FILE, "w") as out_f:
    for qa_file in qa_files:
        label = Path(qa_file).stem

        # Read this source's QA pairs
        items = []
        with open(qa_file) as f:
            for line in f:
                items.append(json.loads(line))

        if not items:
            print(f"  {label:25s}    0 QA →    0 conversations (empty)")
            continue

        # Group by chunk_key (related QAs stay together)
        groups = defaultdict(list)
        for item in items:
            groups[item["chunk_key"]].append(item)

        # Build multi-turn conversations with unified system prompt
        source_count = 0
        for _, group_items in groups.items():
            random.shuffle(group_items)
            for i in range(0, len(group_items), TURNS_PER_CONVERSATION):
                batch = group_items[i:i + TURNS_PER_CONVERSATION]
                if len(batch) < 2:
                    continue
                conv = {"conversations": [
                    {"from": "system", "value": UNIFIED_SYSTEM_PROMPT}
                ]}
                for qa in batch:
                    conv["conversations"].append({"from": "human", "value": qa["question"]})
                    conv["conversations"].append({"from": "gpt", "value": qa["answer"]})
                if quality_check(conv):
                    out_f.write(json.dumps(conv) + "\n")
                    source_count += 1
                else:
                    template_filtered_convs += 1

        total_convs += source_count
        print(f"  {label:25s} {len(items):>5} QA → {source_count:>4} conversations")
        del items, groups
        gc.collect()

print(f"\n✓ Saved {total_convs:,} conversations to:")
print(f"  {OUTPUT_FILE}")
print(f"  ({os.path.getsize(OUTPUT_FILE) / 1024 / 1024:.1f} MB)")
if template_filtered_convs:
    print(f"  (filtered {template_filtered_convs} conversations with template answers)")

# Shuffle the output file for better training
import subprocess
subprocess.run(["shuf", OUTPUT_FILE, "-o", OUTPUT_FILE])
print(f"  ✓ Shuffled output file")

## 7. Verify

In [ ]:
# Verify by streaming from disk — no need to hold all conversations in RAM
total_turns = 0
total_convs_verify = 0
sample_convs = []

with open(OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        conv = json.loads(line)
        total_convs_verify += 1
        total_turns += len(conv["conversations"]) - 1

        # Reservoir sample: keep up to 4 random conversations
        if len(sample_convs) < 4:
            sample_convs.append(conv)
        else:
            j = random.randint(0, line_num)
            if j < 4:
                sample_convs[j] = conv

        del conv

print(f"{'='*50}")
print(f"TOTAL: {total_convs_verify:,} conversations, {total_turns:,} turns ({total_turns//2:,} QA pairs)")
print(f"Persona: adam_eve_narrator (Books I & II combined)")

# Sample conversations
print(f"\n{'='*50}")
print("SAMPLE CONVERSATIONS:\n")
for conv in sample_convs:
    print(f"{'─'*50}")
    for msg in conv["conversations"]:
        role = msg["from"].upper()
        text = msg["value"][:250] + ("..." if len(msg["value"]) > 250 else "")
        print(f"[{role}] {text}\n")

del sample_convs
gc.collect()

print(f"\n✓ Data ready for training. Load this file in your training notebook:")
print(f"  {OUTPUT_FILE}")